In [ ]:
# Phase 7 v3 screen, fold 0: does prep v3's nineteen-slot table beat prep v2's
# six under c2's config? Both arms run the packed model -- absent slots never
# reach the backbone -- so the table is the only variable, measured in one session.
import glob, os, shutil, sys, time

GIT_SHA = '62e87a4'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import inspect
import torch

assert torch.cuda.is_available()
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{major}{minor}')
assert (major, minor) >= (7, 0), 'need T4, not P100'
device = 'cuda'

from knee.dataset import PreppedSlotDataset, _SLOT_PLANES
from knee.dicom import SLOTS, SLOTS_V3
from knee.model import KneeModel
from knee.reports import confidence_from_agreement, report_group_key
from knee.train import (differential_param_groups, make_folds, masked_bce_loss,
                        train_one_epoch)
# The stale-src guard, derived from what this run needs. Without packing the
# arms would differ in BatchNorm statistics as well as the table; without the
# v3 plane lookup the second arm dies on its first study.
REQUIRED = [
    ('KneeModel.__init__', KneeModel.__init__, ('head', 'backbone_kwargs')),
    ('KneeModel._slot_features', KneeModel._slot_features, ('mask',)),
    ('make_folds', make_folds, ('groups',)),
    ('PreppedSlotDataset.__init__', PreppedSlotDataset.__init__, ('weights_df',)),
]
for label, fn, needed in REQUIRED:
    have = set(inspect.signature(fn).parameters)
    missing = [n for n in needed if n not in have]
    assert not missing, (label + ' is missing ' + repr(missing) +
                         ' -- rsna-knee-src is stale. Re-version it before pushing.')
assert hasattr(KneeModel, '_backbone_features'), 'rsna-knee-src predates slot packing'
assert all(name in _SLOT_PLANES for name, *_ in SLOTS_V3), 'rsna-knee-src cannot load v3 slots'
print('src carries slot packing and the v3 loader | GIT_SHA', GIT_SHA)

In [ ]:
# GATE -- both corpora, asserted from their shard manifests. The v2 and v3 prep
# kernels write identically named files, so each corpus is told apart by the
# kernel it is mounted from, and each must cover the corpus on its own.
import numpy as np
import pandas as pd

def mounted(tag):
    dirs = sorted(d for d in glob.glob('/kaggle/input/**/prepped', recursive=True) if tag in d)
    manifests = sorted(p for p in glob.glob('/kaggle/input/**/prep_manifest_shard*.csv',
                                            recursive=True) if tag in p)
    failed = sorted(p for p in glob.glob('/kaggle/input/**/prep_failed_shard*.csv',
                                         recursive=True) if tag in p)
    assert len(dirs) == 4 and len(manifests) == 4, (
        f'{tag}: {len(dirs)} prepped dirs and {len(manifests)} manifests, expected 4 each')
    npz = {}
    for d in dirs:
        for f in os.listdir(d):
            if f.endswith('.npz'):
                npz[f[:-4]] = os.path.join(d, f)
    manifest = pd.concat([pd.read_csv(p) for p in manifests])
    n_failed = sum(len(pd.read_csv(p)) for p in failed)
    print(f'{tag}: {len(npz)} artifacts, {len(manifest)} manifest rows, {n_failed} failed')
    assert len(npz) == len(manifest), f'{tag}: artifact count and manifest disagree'
    assert len(npz) + n_failed == 4407, f'{tag}: shards do not cover the corpus'
    return npz, manifest

def link_root(npz, root):
    # one directory per corpus by symlink, as every screen does; costs no space
    os.makedirs(root, exist_ok=True)
    for uid, path in npz.items():
        dst = os.path.join(root, f'{uid}.npz')
        if not os.path.exists(dst):
            os.symlink(path, dst)
    return root

npz_v2, manifest_v2 = mounted('phase6-prep')
npz_v3, manifest_v3 = mounted('phase7-prep')
NPZ_V2 = link_root(npz_v2, '/kaggle/working/prepped_v2')
NPZ_V3 = link_root(npz_v3, '/kaggle/working/prepped_v3')

# Both arms train and score on the studies both tables hold, so the pair is exact.
# Named uid_to_npz because the shared folds-and-labels cell filters gold by it.
uid_to_npz = set(npz_v2) & set(npz_v3)
print(f'studies in both corpora: {len(uid_to_npz)}')

V2_SLOTS = [name for name, *_ in SLOTS]
V3_SLOTS = [name for name, *_ in SLOTS_V3]
for tag, manifest, names in (('v2', manifest_v2, V2_SLOTS), ('v3', manifest_v3, V3_SLOTS)):
    fill = manifest.slots_filled.fillna('').str.get_dummies(sep='|')
    strangers = set(fill.columns) - set(names)
    assert not strangers, f'{tag} manifest files series under slots outside its table: {strangers}'
    print(f'\n{tag}: mean slots/study {manifest.n_slots_filled.mean():.2f} of {len(names)}')
    for name in names:
        n = int(fill[name].sum()) if name in fill else 0
        print(f'  {name:13s} {n:5d}  {n / len(manifest):6.1%}')

In [ ]:
# Folds v3 and labels. Folds are regenerated here rather than uploaded, and
# pinned by hash: the experiment discipline requires one frozen fold assignment
# across every Phase 6 run, and a silent drift would make every paired delta
# meaningless without failing anything.
import hashlib

from knee.infer import LABEL_COLUMNS
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.train import Timer, evaluate, log_experiment, train_val_split
from torch.utils.data import DataLoader

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

groups = {u: report_group_key(r)
          for u, r in zip(train_df['StudyInstanceUID'].astype(str), train_df['Report'])}
folds = make_folds(all_uids, n_folds=5, seed=0, groups=groups)
FOLDS_PRIMARY_V3_SHA256 = 'f1d6ba7c341f8d2e82241cb4ddd5ca5131b0c6d2033eb6d2e43bcb597cf12868'
assert hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()
).hexdigest() == FOLDS_PRIMARY_V3_SHA256, 'fold assignment drifted from folds_primary_v3.csv'
n_straddle = sum(1 for _, g in pd.DataFrame(
    {'g': [groups[u] for u in all_uids], 'f': [folds[u] for u in all_uids]}
).groupby('g') if g.f.nunique() > 1)
print(f'folds v3 verified; {n_straddle} report groups straddle folds (v2 had 47)')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_df = train_df[train_df[gold_cols].notna().any(axis=1)]
holdout = frozenset(gold_df['StudyInstanceUID'].astype(str))
print(f'{len(holdout)} gold studies held out of both sides of every split')

# Step 3 re-baselines on the same label source Phase 5 trained on; swapping the
# label target is c0, a separate cell, so the two changes stay attributable.
pseudo = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0])
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]]
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS

# Two frames, as Phase 5 used: soft scores are the training target, but AUC is
# scored against a binarized one -- roc_auc_score rejects a continuous y_true,
# and the soft target is a confidence, not a label.
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (labels_all[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
print(f'{len(labels_all)} pseudo-labelled studies (soft for training, binarized for scoring)')

# c0's alternative target. steven_v4 is the best single public source on the 58
# gold studies (0.8927 vs our 0.8616, paired delta +0.031 [+0.004, +0.058]).
#
# NOT the 5-source rank blend, which scored higher on gold (0.8976): percentile
# ranks are uniform by construction, so every label's target mean is exactly
# 0.500 and all prevalence information is destroyed -- Fracture would be trained
# at 0.5 against a real rate of 0.014. AUC never notices, because AUC reads
# order only. That is exactly why rank-averaging is right for combining
# predictions at submission time and wrong for building a target.
LABEL_SOURCES = {'ours_qwen3_4b': (labels_all, labels_eval)}
steven = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
if steven:
    alt = pd.read_csv(steven[0])
    alt['StudyInstanceUID'] = alt['StudyInstanceUID'].astype(str)
    alt = alt[['StudyInstanceUID'] + LABEL_COLUMNS]
    alt_eval = alt.copy()
    alt_eval[LABEL_COLUMNS] = (alt[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
    LABEL_SOURCES['steven_v4'] = (alt, alt_eval)
    rate = alt[LABEL_COLUMNS].mean()
    print(f'steven_v4 loaded: {len(alt)} studies, positive rate '
          f'{rate.min():.3f}-{rate.max():.3f} (ours: '
          f'{labels_all[LABEL_COLUMNS].mean().min():.3f}-'
          f'{labels_all[LABEL_COLUMNS].mean().max():.3f})')
print('label sources available:', list(LABEL_SOURCES))

# Confidence weights, derived here rather than mounted as a CSV so the derivation
# is reproducible where it is used. Four INDEPENDENT labellings vote: ours,
# steven_v4, pilkwang_v2, lixin73. Steven ships full/v2/v4_blend but those are one
# author's successive versions and would triple-count that author's reading;
# yunusgmsoy is excluded for carrying gold values on the 58 held-out rows.
#
# CAVEAT measured on 2026-09-07 and worth re-reading before believing a positive
# result: agreement is not accuracy. Across the 12 labels, disagreement correlates
# +0.40 (n=12, p=0.20) with steven_v4's gold per-label AUC -- the wrong sign.
# Synovitis is the extreme: near-perfect source agreement (1.2% disputed) and the
# worst-labelled of the twelve. confidence_from_agreement normalises to mean 1 PER
# LABEL, which removes exactly that between-label axis, so what is being tested
# here is strictly within-label. The prior is weaker than it looks.
def _conf_source(pattern, use_conf):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not hits:
        return None
    df = pd.read_csv(hits[0])
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    out = pd.DataFrame({'StudyInstanceUID': df['StudyInstanceUID']})
    for lab in LABEL_COLUMNS:
        if lab not in df.columns:
            return None
        v = pd.to_numeric(df[lab], errors='coerce')
        conf_col = f'{lab}__conf'
        if use_conf and conf_col in df.columns:
            c = pd.to_numeric(df[conf_col], errors='coerce').clip(0, 1).fillna(0.5)
            v = v * c + (1 - v) * (1 - c)
        out[lab] = v
    return out.set_index('StudyInstanceUID')

_votes = [labels_all.set_index('StudyInstanceUID')[LABEL_COLUMNS]]
for _pat, _uc in (('llm_labels_v4_blend.csv', False),
                  ('report_labels_v2.csv', True),
                  ('report_labels_gpt56sol.csv', True)):
    _v = _conf_source(_pat, _uc)
    if _v is not None:
        _votes.append(_v)

# Below three voices, spread is not a meaningful agreement signal -- fail loudly
# rather than train a "weighted" arm that is silently the unweighted one.
assert len(_votes) >= 4, f'only {len(_votes)} label sources mounted; need 4'
CONFIDENCE = confidence_from_agreement(_votes, LABEL_COLUMNS).reset_index()
CONFIDENCE = CONFIDENCE.rename(columns={CONFIDENCE.columns[0]: 'StudyInstanceUID'})
print(f'confidence weights: {len(CONFIDENCE)} studies from {len(_votes)} sources; '
      f'share below 0.5 per label '
      f'{(CONFIDENCE[LABEL_COLUMNS] < 0.5).mean().round(2).to_dict()}')

# The arbiter for c0. Scoring each model against its OWN label source would be
# circular -- that measures how learnable a label set is, not which one trains a
# better model, and each source would be graded by its own marker. The 58 gold
# studies are rubric-graded from images, excluded from every training split, and
# identical for both arms, so they are the only unbiased comparison available
# when the target itself is the variable. n=58 is thin (MCL has 9 positives) and
# the per-label numbers are directional only; the macro is the read.
gold_labels = gold_df[['StudyInstanceUID'] + LABEL_COLUMNS].copy()
gold_labels['StudyInstanceUID'] = gold_labels['StudyInstanceUID'].astype(str)
gold_uids = [u for u in gold_labels['StudyInstanceUID'] if u in uid_to_npz]
print(f'gold arbiter: {len(gold_uids)} of 58 studies have a prepped artifact')

In [ ]:
# The runs: one config at a time, each logged, a time guard between arms.
EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
# Interpolated from results/experiments.csv at build time: log_experiment selects
# by this header, and a hand-copied one that drifts drops a column silently.
_HEADER = 'date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,macro_auc,paired_delta,train_minutes,inference_seconds,promoted,gold_macro'
with open(EXPERIMENTS_CSV, 'w') as f:
    print(_HEADER, file=f)

CONFIGS = [
    # name, npz root, slots, n_groups, out_size, epochs, batch, head, backbone
    ('v3s_c2_6slot_v2', NPZ_V2, V2_SLOTS, 5, 224, 8, 8, 'slot_attention', 'efficientnet_b0'),
    ('v3s_c2_19slot_v3', NPZ_V3, V3_SLOTS, 5, 224, 8, 8, 'slot_attention', 'efficientnet_b0'),
]
HYPOTHESIS = {
    'v3s_c2_6slot_v2':
        'v3 screen control: c2 (efficientnet_b0, slot attention, 8 epochs, steven_v4) on '
        'the prep v2 six-slot table with absent slots packed out of the backbone. Pairs '
        'against the 19-slot arm in the same session; read unpaired against the unpacked '
        'c2 confirm fold-0 row for what packing alone did.',
    'v3s_c2_19slot_v3':
        'v3 screen: the same c2 config on the prep v3 table, plane x fat suppression x '
        'T1/PD/T2/UNK. The census found every v2 STRUCT slot mixing contrasts; the risk is '
        'sparsity, mean 5.04 of 19 filled. Worth a 5-fold confirm only if the paired val '
        'CI lower bound is above zero.',
}
SOURCE = 'steven_v4'
FOLD = 0
# c2 took 47 min per fold unpacked and packing only removes work. The guard stops
# the second arm starting if the first overran, rather than losing both to a kill.
TIME_BUDGET_S = 3 * 3600
t_start = time.time()

def loader(root, uids, labels_df, cfg_slots, n_groups, out_size, batch,
           shuffle=False, augment=False):
    ds = PreppedSlotDataset(uids, root, labels_df=labels_df, n_groups=n_groups,
                            out_size=out_size, slots=cfg_slots, augment=augment)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, num_workers=2)

train_labels, eval_labels = LABEL_SOURCES[SOURCE]
tr_uids, va_uids = train_val_split(folds, val_fold=FOLD, exclude_uids=holdout)
tr_uids = [u for u in tr_uids if u in uid_to_npz]
va_uids = [u for u in va_uids if u in uid_to_npz]

results = []
for name, root, cfg_slots, n_groups, out_size, epochs, batch, head, backbone in CONFIGS:
    if time.time() - t_start > TIME_BUDGET_S:
        print(f'time budget spent, skipping {name}')
        break
    print(f'\n=== {name}: {len(tr_uids)} train / {len(va_uids)} val, '
          f'{len(cfg_slots)} slots x {n_groups} groups at {out_size}px, {epochs} epochs')

    train_loader = loader(root, tr_uids, train_labels, cfg_slots, n_groups, out_size, batch,
                          shuffle=True, augment=True)
    val_loader = loader(root, va_uids, eval_labels, cfg_slots, n_groups, out_size, batch)
    model = KneeModel(backbone_name=backbone, num_labels=12, pretrained=True,
                      head=head).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler('cuda')
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=3e-4, epochs=epochs, steps_per_epoch=len(train_loader))

    # Counted over the first epoch: images the backbone actually runs per batch,
    # against what it would run unpacked. On real fill this is the claim that
    # makes nineteen slots cost about what six do.
    seen = []
    hook = model.backbone.register_forward_pre_hook(lambda _m, args: seen.append(args[0].shape[0]))
    t0 = time.time()
    for epoch in range(epochs):
        loss = train_one_epoch(model, train_loader, optimizer, device=device,
                               scaler=scaler, scheduler=scheduler)
        if epoch == 0:
            hook.remove()
            print(f'  backbone images per batch {np.mean(seen):.1f}, '
                  f'unpacked would be {batch * len(cfg_slots) * n_groups}')
        print(f'  epoch {epoch + 1}/{epochs} loss {loss:.4f} '
              f'({(time.time() - t0) / 60:.1f} min)')
    minutes = (time.time() - t0) / 60

    y_true, y_pred = evaluate(model, val_loader, device=device)
    aucs = per_label_auc(y_true, y_pred)
    macro = macro_auc(y_true, y_pred)

    gold_loader = loader(root, gold_uids, gold_labels, cfg_slots, n_groups, out_size, batch)
    g_true, g_pred = evaluate(model, gold_loader, device=device)
    gold_macro = macro_auc(g_true, g_pred)
    np.save(f'/kaggle/working/gold_pred_{name}.npy', g_pred)
    np.save(f'/kaggle/working/gold_true_{name}.npy', g_true)

    # Which slot each diagnosis reads, over the gold studies. Averaged only over
    # the studies where the slot is present: folding in absent studies' zero
    # weight would make a rarely filled slot look ignored.
    model.eval()
    chunks, masks = [], []
    with torch.no_grad():
        for images, m, _, _ in gold_loader:
            chunks.append(model.attention_weights(images.to(device), m.to(device)).cpu().numpy())
            masks.append(m.numpy())
    attn, present = np.concatenate(chunks), np.concatenate(masks)
    np.save(f'/kaggle/working/attn_{name}.npy', attn)
    mean_attn = attn.sum(axis=0) / np.maximum(present.sum(axis=0), 1)[:, None]
    print('    attention by slot (rows) and diagnosis (columns), where present on gold:')
    print('      ' + ' ' * 14 + ' '.join(f'{label[:6]:>6s}' for label in LABEL_COLUMNS))
    for si, slot in enumerate(cfg_slots):
        print(f'      {slot:13s} ' + ' '.join(f'{mean_attn[si, li]:6.3f}'
                                             for li in range(len(LABEL_COLUMNS)))
              + f'   n={int(present[:, si].sum())}')

    print(f'  {name}: val macro {macro:.4f}, GOLD macro {gold_macro:.4f} in {minutes:.1f} min')
    for label, auc in zip(LABEL_COLUMNS, aucs):
        print(f'    {label:20s} {auc:.4f}')
    results.append({'config': name, 'macro_auc': macro, 'gold_macro': gold_macro,
                    'minutes': minutes, 'final_loss': loss, 'n_val': len(va_uids),
                    **dict(zip(LABEL_COLUMNS, aucs))})
    log_experiment(EXPERIMENTS_CSV, git_sha=GIT_SHA, config_hash=name,
                   hypothesis=HYPOTHESIS[name], fold_set=f'primary_v3_fold{FOLD}',
                   seed=0, per_label_auc={l: float(a) for l, a in zip(LABEL_COLUMNS, aucs)},
                   macro_auc=float(macro), paired_delta=float('nan'),
                   train_minutes=minutes, inference_seconds=float('nan'), promoted=False,
                   gold_macro=float(gold_macro))
    np.save(f'/kaggle/working/oof_{name}.npy', y_pred)
    np.save(f'/kaggle/working/y_{name}.npy', y_true)
    torch.save(model.state_dict(), f'/kaggle/working/{name}.pt')
    # freed only after every consumer of `model` has run
    del model, optimizer, scaler, scheduler
    torch.cuda.empty_cache()

In [ ]:
# Results. Val (same target, same studies, both arms) decides; gold (n=58) is
# reported beside it. The 19-slot table is worth a 5-fold confirm only if the
# paired val CI excludes zero on the positive side -- a gate into the confirm,
# not a result: fold 0 has misled before.
C2_UNPACKED_FOLD0 = 0.8571   # c2 confirm, fold 0, backbone saw every absent slot

table = pd.DataFrame(results)
table.to_csv('/kaggle/working/phase7_screen_v3_fold0.csv', index=False)
print(table[['config', 'macro_auc', 'gold_macro', 'final_loss', 'minutes']].to_string(index=False))

def saved(kind, config):
    return np.load(f'/kaggle/working/{kind}_{config}.npy')

if len(results) == 2:
    six, nineteen = results[0]['config'], results[1]['config']
    print()
    for tier, truth, pred, n in (('val ', 'y', 'oof', results[0]['n_val']),
                                 ('gold', 'gold_true', 'gold_pred', 58)):
        y = saved(truth, six)
        assert np.array_equal(np.nan_to_num(y, nan=-1), np.nan_to_num(saved(truth, nineteen), nan=-1))
        p6, p19 = saved(pred, six), saved(pred, nineteen)
        delta, lo, hi = paired_macro_auc_delta(y, p6, p19)
        verdict = 'CI excludes zero' if lo > 0 or hi < 0 else 'CI spans zero'
        print(f'{tier} (n={n}) 19 slots minus 6:        {delta:+.4f} [{lo:+.4f}, {hi:+.4f}]  {verdict}')
        delta, lo, hi = paired_macro_auc_delta(y, p6, (p6 + p19) / 2)
        print(f'{tier} (n={n}) both tables averaged minus 6: {delta:+.4f} [{lo:+.4f}, {hi:+.4f}]')

print()
print(f"packing alone, UNPAIRED (different session, ~0.0065 run-to-run noise): "
      f"6 slots packed {results[0]['macro_auc']:.4f} vs c2 confirm fold 0 unpacked "
      f"{C2_UNPACKED_FOLD0:.4f}, 47.4 min")